## Testing DAGGER

### MountainCar

In [ ]:
import tempfile
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from imitation.algorithms import bc
from imitation.algorithms.dagger import SimpleDAggerTrainer
from stable_baselines3.common.env_util import make_vec_env

from stable_baselines3 import PPO, DDPG, SAC, TD3
from sb3_contrib import TRPO
from stable_baselines3.common.evaluation import evaluate_policy

PROJECT_ROOT = Path.cwd().parent
#BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
cwd = Path.cwd()
current = cwd
while not (current / "code" / "baseline_code").exists() and current != current.parent:
    current = current.parent

PROJECT_ROOT = current
BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
print("CWD:", cwd)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASELINE_CODE:", BASELINE_CODE)

sys.path.insert(0, str(BASELINE_CODE))
print(Path.cwd())
from baseline_enviroments.MountainCar import make_continuous_mountaincar


# ----- Paths -----
TEACHER_DIR = BASELINE_CODE / "baseline_models" / "mountaincar"
DAGGER_OUT_DIR = BASELINE_CODE / "dagger_models" / "mountaincar"
DAGGER_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- Map filename prefix -> loader -----
ALGOS = {
    "PPO": PPO,
    "DDPG": DDPG,
    "SAC": SAC,
    "TD3": TD3,
    "TRPO": TRPO,
}

# ----- Shared -----
rng = np.random.default_rng(0)
ENV_FACTORY = make_continuous_mountaincar

# DAgger settings (adjust)
DAGGER_TIMESTEPS = 50_000      # total timesteps for SimpleDAggerTrainer
N_EVAL_EPISODES = 10

results = []

# Find teacher zips like "PPO_mountaincar.zip"
teacher_paths = sorted(TEACHER_DIR.glob("*_mountaincar.zip"))
if not teacher_paths:
    raise FileNotFoundError(f"No teacher models found in: {TEACHER_DIR}")

for teacher_zip in teacher_paths:
    algo_name = teacher_zip.name.split("_")[0]  # "PPO" from "PPO_mountaincar.zip"
    if algo_name not in ALGOS:
        print(f"Skipping unknown algo file: {teacher_zip.name}")
        continue

    Algo = ALGOS[algo_name]
    print(f"\n=== DAgger for {algo_name} teacher: {teacher_zip.name} ===")

    # Create venv for DAgger
    venv = make_vec_env(ENV_FACTORY, n_envs=1)

    # Load teacher; env=None avoids observation-space equality checks
    expert = Algo.load(str(teacher_zip), env=None)

    # BC trainer (student policy)
    bc_trainer = bc.BC(
        observation_space=venv.observation_space,
        action_space=venv.action_space,
        rng=rng,
    )

    with tempfile.TemporaryDirectory(prefix=f"dagger_{algo_name}_") as tmpdir:
        dagger_trainer = SimpleDAggerTrainer(
            venv=venv,
            scratch_dir=tmpdir,
            expert_policy=expert,
            bc_trainer=bc_trainer,
            rng=rng,
        )

        dagger_trainer.train(total_timesteps=DAGGER_TIMESTEPS)

        # Evaluate student
        eval_env = make_vec_env(ENV_FACTORY, n_envs=1)
        mean_reward, std_reward = evaluate_policy(
            dagger_trainer.policy, eval_env, n_eval_episodes=N_EVAL_EPISODES
        )

        # Save the learned DAgger policy (imitation policies use .save)
        out_path = DAGGER_OUT_DIR / f"DAgger_{algo_name}_mountaincar"
        dagger_trainer.policy.save(str(out_path))
        print(f"Saved DAgger student to: {out_path}")

        results.append(
            {
                "teacher_algo": algo_name,
                "teacher_file": teacher_zip.name,
                "dagger_timesteps": DAGGER_TIMESTEPS,
                "mean_reward": mean_reward,
                "std_reward": std_reward,
                "saved_student": str(out_path),
            }
        )

        eval_env.close()
    venv.close()

df = pd.DataFrame(results).sort_values("mean_reward", ascending=False)
print(df)
#df.to_csv(DAGGER_OUT_DIR / "dagger_results.csv", index=False)
print(f"\nWrote results to: {DAGGER_OUT_DIR / 'dagger_results.csv'}")


CWD: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/notebooks
PROJECT_ROOT: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale
BASELINE_CODE: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code
/Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/notebooks

=== DAgger for DDPG teacher: DDPG_mountaincar.zip ===


Saving the dataset (1/1 shards): 100%|██████████| 1/1 [00:00<00:00, 256.47 examples/s]
0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00142 |
|    entropy        | 1.42     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 68.5     |
|    loss           | 0.918    |
|    neglogp        | 0.919    |
|    prob_true_act  | 0.399    |
|    samples_so_far | 32       |
| rollout/          |          |
|    return_max     | -48.6    |
|    return_mean    | -51.4    |
|    return_min     | -53.8    |
|    return_std     | 1.71     |
--------------------------------


372batch [00:02, 166.98batch/s]
Saving the dataset (1/1 shards): 100%|██████████| 1/1 [00:00<00:00, 272.43 examples/s]
0batch [00:00, ?batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 0        |
|    ent_loss       | -0.00105 |
|    entropy        | 1.05     |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 68.5     |
|    loss           | 0.546    |
|    neglogp        | 0.547    |
|    prob_true_act  | 0.579    |
|    samples_so_far | 32       |
| rollout/          |          |
|    return_max     | -34.7    |
|    return_mean    | -35.9    |
|    return_min     | -37.3    |
|    return_std     | 0.847    |
--------------------------------


...

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 500      |
|    ent_loss       | 0.00549  |
|    entropy        | -5.49    |
|    epoch          | 0        |
|    l2_loss        | 0        |
|    l2_norm        | 76.2     |
|    loss           | -5.38    |
|    neglogp        | -5.38    |
|    prob_true_act  | 269      |
|    samples_so_far | 16032    |
| rollout/          |          |
|    return_max     | -0.0913  |
|    return_mean    | -0.0915  |
|    return_min     | -0.0919  |
|    return_std     | 0.000225 |
--------------------------------


993batch [00:04, 330.71batch/s]

--------------------------------
| batch_size        | 32       |
| bc/               |          |
|    batch          | 1000     |
|    ent_loss       | 0.00549  |
|    entropy        | -5.49    |
|    epoch          | 1        |
|    l2_loss        | 0        |
|    l2_norm        | 76       |
|    loss           | -5.48    |
|    neglogp        | -5.48    |
|    prob_true_act  | 282      |
|    samples_so_far | 32032    |
| rollout/          |          |
|    return_max     | -0.0918  |
|    return_mean    | -0.0922  |
|    return_min     | -0.0927  |
|    return_std     | 0.000298 |
--------------------------------


1120batch [00:06, 171.73batch/s]


KeyboardInterrupt: 

## Acrobot

In [24]:
import tempfile
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from imitation.algorithms import bc
from imitation.algorithms.dagger import SimpleDAggerTrainer

from stable_baselines3 import PPO, DDPG, SAC, TD3
from sb3_contrib import TRPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env

# ----------------------------
# Project path handling
# ----------------------------
cwd = Path.cwd()
current = cwd
while not (current / "code" / "baseline_code").exists() and current != current.parent:
    current = current.parent

PROJECT_ROOT = current
BASELINE_CODE = PROJECT_ROOT / "code" / "baseline_code"
sys.path.insert(0, str(BASELINE_CODE))

print("PROJECT_ROOT:", PROJECT_ROOT)

# ----------------------------
# Import Acrobot wrapper
# ----------------------------
from baseline_enviroments.acrobot_env import make_continuous_acrobot

# ----------------------------
# Paths
# ----------------------------
TEACHER_DIR = BASELINE_CODE / "baseline_models" / "acrobot"
DAGGER_OUT_DIR = BASELINE_CODE / "dagger_models" / "acrobot"
DAGGER_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Teacher dir:", TEACHER_DIR)
print("Teacher files:", [p.name for p in TEACHER_DIR.glob("*.zip")])

# ----------------------------
# Algorithm map
# ----------------------------
ALGOS = {
    "PPO": PPO,
    "DDPG": DDPG,
    "SAC": SAC,
    "TD3": TD3,
    "TRPO": TRPO,
}

rng = np.random.default_rng(0)
ENV_FACTORY = make_continuous_acrobot

DAGGER_TIMESTEPS = 50_000
N_EVAL_EPISODES = 10

results = []

teacher_paths = sorted(TEACHER_DIR.glob("*.zip"))
if not teacher_paths:
    raise FileNotFoundError(f"No teacher models found in: {TEACHER_DIR}")

for teacher_zip in teacher_paths:
    algo_name = teacher_zip.name.split("_")[0]

    if algo_name not in ALGOS:
        print(f"Skipping unknown file: {teacher_zip.name}")
        continue

    Algo = ALGOS[algo_name]
    print(f"\n=== DAgger for {algo_name} teacher ===")

    # Create training environment
    venv = make_vec_env(ENV_FACTORY, n_envs=1)

    # Load teacher
    expert = Algo.load(str(teacher_zip), env=None)

    # Student BC trainer
    bc_trainer = bc.BC(
        observation_space=venv.observation_space,
        action_space=venv.action_space,
        rng=rng,
    )

    with tempfile.TemporaryDirectory(prefix=f"dagger_{algo_name}_") as tmpdir:
        dagger_trainer = SimpleDAggerTrainer(
            venv=venv,
            scratch_dir=tmpdir,
            expert_policy=expert,
            bc_trainer=bc_trainer,
            rng=rng,
        )

        dagger_trainer.train(total_timesteps=DAGGER_TIMESTEPS)

        # Evaluate student
        eval_env = make_vec_env(ENV_FACTORY, n_envs=1)
        mean_reward, std_reward = evaluate_policy(
            dagger_trainer.policy,
            eval_env,
            n_eval_episodes=N_EVAL_EPISODES,
        )

        # Save student
        out_path = DAGGER_OUT_DIR / f"DAgger_{algo_name}_acrobot"
        dagger_trainer.policy.save(str(out_path))
        print(f"Saved DAgger student to: {out_path}")

        results.append({
            "teacher_algo": algo_name,
            "mean_reward": mean_reward,
            "std_reward": std_reward,
        })

        eval_env.close()
    venv.close()

df = pd.DataFrame(results).sort_values("mean_reward", ascending=False)
print(df)
df.to_csv(DAGGER_OUT_DIR / "dagger_results.csv", index=False)
print("\nResults saved.")


PROJECT_ROOT: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale
Teacher dir: /Users/elisalaegsgaard/Desktop/Speciale/GGSpeciale/code/baseline_code/baseline_models/acrobot
Teacher files: ['PPO_acrobot.zip', 'TRPO_acrobot.zip', 'DDPG_acrobot.zip', 'SAC_acrobot.zip', 'TD3_acrobot.zip']

=== DAgger for DDPG teacher ===


ValueError: <class 'numpy.random._pcg64.PCG64'> is not a known BitGenerator module.